# Chatterbox TTS + Turbo — Colab runner

Runs the app on a free Colab GPU. Nothing installs on your own machine.

**Per session:** Runtime → *Change runtime type* → GPU, then Runtime → *Run all*.
First run takes ~3–5 min (pip) + ~2–4 min (model download from Hugging Face).
The `https://xxxx.gradio.live` link appears in the last cell's output.

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi -L || echo 'NO GPU — Runtime > Change runtime type > GPU, then rerun'

In [ ]:
# 2. Get the code from GitHub (clone first time, pull afterwards).
# Idempotent: safe to re-run this cell any number of times in the same
# session -- always resolves to the same absolute path instead of `%cd`
# compounding into nested repo-in-a-repo folders on repeat runs.
import os
REPO_URL = 'https://github.com/ducp507/chatterbox-colab.git'  # <-- change if you rename the repo
REPO_NAME = REPO_URL.rstrip('/').split('/')[-1][:-4]
REPO_DIR = f"/content/{REPO_NAME}"
if not os.path.isdir(REPO_DIR):
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    %cd {REPO_DIR}
    !git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# 3. Install deps (uses Colab's preinstalled torch/torchaudio/numpy)
!pip install -q -r requirements-colab.txt
# If you hit an import error below, run instead:  !pip install -q -r requirements.txt

In [ ]:
# 4. (optional) cache model weights on Google Drive so future sessions skip the re-download
#    Uncomment the 3 lines, approve the Drive popup once.
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ['HF_HOME'] = '/content/drive/MyDrive/hf-cache'
# os.makedirs(os.environ['HF_HOME'], exist_ok=True)

In [ ]:
# 5. Launch — wait for the  Running on public URL: https://....gradio.live  line.
# This blocks the kernel (the server never exits) -- if you need to run another
# cell (a debug snippet, pushing a cloned voice, etc.), stop this cell first,
# run the other one, then come back and run this cell again to relaunch.
!GRADIO_SHARE=1 TOKENIZERS_PARALLELISM=false python app.py

### Keep a cloned voice between sessions
Voices you clone land in `modules/voice_samples/` which is wiped when the VM recycles.
To keep one, run in a new cell (needs a GitHub token with repo write):
```python
!git config user.email you@example.com && git config user.name you
!git add modules/voice_samples/ && git commit -m 'add voice' && git push https://TOKEN@github.com/ducp507/chatterbox-colab.git
```
Or just download the `.wav` from the file browser on the left.